<a href="https://colab.research.google.com/github/mybenkhadda/enveda-kaggle/blob/main/notebooks/01_colab_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Colab inference from Google Drive filenames only

This notebook runs the frozen `v2-A7` / `V1_TL_1K_TESTSIM_STRICT` / `MOST_CONFIDENT_SPECTRUM` pipeline directly from the named files in `MyDrive/EnvedaCASMI`.

There are no Drive file IDs, no bundle copy to local storage, and no file-digest verification. Runtime gates are: required filenames exist, frozen model/aggregator identity is correct, aggregation metadata is internally consistent, the functional self-test has exact downstream parity, and submission validation passes.


In [1]:
# 1. Mount Drive and define exact filenames
from google.colab import drive
drive.mount('/content/drive')

from datetime import datetime, timezone
from pathlib import Path
import json, os, sys, shutil, subprocess

REPO_URL = 'https://github.com/mybenkhadda/enveda-kaggle.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/enveda-kaggle')

DRIVE_ROOT = Path('/content/drive/MyDrive/EnvedaCASMI').resolve()
BUNDLE_ROOT = (DRIVE_ROOT / 'bundle').resolve()
COMPETITION_ROOT = (DRIVE_ROOT / 'competition').resolve()
DRIVE_RESULTS = (DRIVE_ROOT / 'results').resolve()

TEST_PATH = COMPETITION_ROOT / 'test.parquet'
SAMPLE_SUBMISSION_PATH = COMPETITION_ROOT / 'sample_submission.csv'
CONFIG_PATH = BUNDLE_ROOT / 'config.json'
MANIFEST_PATH = BUNDLE_ROOT / 'manifest.json'
MODEL_INFO_PATH = BUNDLE_ROOT / 'models' / 'model_info.json'

DEVICE_PREFERENCE = 'gpu'
REQUIRE_GPU = False
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT_ROOT = Path('/content/casmi_runtime/output').resolve()
WORK_DIR = OUTPUT_ROOT / RUN_ID
WORK_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

print('Bundle:', BUNDLE_ROOT)
print('Test:', TEST_PATH)
print('Sample:', SAMPLE_SUBMISSION_PATH)
print('Work:', WORK_DIR)


Mounted at /content/drive
Bundle: /content/drive/MyDrive/EnvedaCASMI/bundle
Test: /content/drive/MyDrive/EnvedaCASMI/competition/test.parquet
Sample: /content/drive/MyDrive/EnvedaCASMI/competition/sample_submission.csv
Work: /content/casmi_runtime/output/20260929T041309Z


In [2]:
# 2. Clone/update the runtime repository and install requirements
if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-colab.txt')], check=True)
print('Repo commit:', REPO_COMMIT)


Repo commit: 9550ca849f0f21c6d4d09bb7af33582d0dda4eff


In [3]:
# 3. Check required bundle/competition filenames
required = [
    MANIFEST_PATH,
    CONFIG_PATH,
    MODEL_INFO_PATH,
    BUNDLE_ROOT / 'ppm_windows.json',
    BUNDLE_ROOT / 'adducts.json',
    BUNDLE_ROOT / '_ref_peaks_int.f8',
    BUNDLE_ROOT / '_ref_peaks_mz.f8',
    BUNDLE_ROOT / 'ref_peaks_int.npy',
    BUNDLE_ROOT / 'ref_peaks_mz.npy',
    BUNDLE_ROOT / 'ref_peaks_offsets.npy',
    BUNDLE_ROOT / 'ref_index_ids.npy',
    BUNDLE_ROOT / 'ref_index_offsets.npy',
    BUNDLE_ROOT / 'ref_meta.parquet',
    BUNDLE_ROOT / 'connectivities.parquet',
    BUNDLE_ROOT / 'structures.parquet',
    BUNDLE_ROOT / 'structures_mass.npy',
    BUNDLE_ROOT / 'aggregation' / 'calibration.json',
    BUNDLE_ROOT / 'aggregation' / 'selected_aggregator.json',
    BUNDLE_ROOT / 'selftest' / 'fixture_manifest.json',
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
]

missing = [p for p in required if not p.exists()]
for p in required:
    print(('OK     ' if p.exists() else 'MISSING'), p.relative_to(DRIVE_ROOT) if DRIVE_ROOT in p.parents else p)
if missing:
    raise FileNotFoundError('Missing required named files:\n' + '\n'.join(map(str, missing)))
print(f'Named files present: {len(required)}/{len(required)}')


OK      bundle/manifest.json
OK      bundle/config.json
OK      bundle/models/model_info.json
OK      bundle/ppm_windows.json
OK      bundle/adducts.json
OK      bundle/_ref_peaks_int.f8
OK      bundle/_ref_peaks_mz.f8
OK      bundle/ref_peaks_int.npy
OK      bundle/ref_peaks_mz.npy
OK      bundle/ref_peaks_offsets.npy
OK      bundle/ref_index_ids.npy
OK      bundle/ref_index_offsets.npy
OK      bundle/ref_meta.parquet
OK      bundle/connectivities.parquet
OK      bundle/structures.parquet
OK      bundle/structures_mass.npy
OK      bundle/aggregation/calibration.json
OK      bundle/aggregation/selected_aggregator.json
OK      bundle/selftest/fixture_manifest.json
OK      competition/test.parquet
OK      competition/sample_submission.csv
Named files present: 21/21


In [4]:
# 4. Load runtime orchestration and check frozen identity from named JSON files
SRC = REPO_DIR / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from casmi_runtime.accelerator import detect_accelerator
from casmi_runtime.config import RuntimeConfig
from casmi_runtime.named_inference import run_named_inference

CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
MODEL_INFO = json.loads(MODEL_INFO_PATH.read_text(encoding='utf-8'))

aggregator_name = CONFIG.get('aggregator_name') or (CONFIG.get('aggregator') or {}).get('name')
assert CONFIG.get('bundle_version') == 'v2-A7', CONFIG.get('bundle_version')
assert CONFIG.get('model_id') == 'V1_TL_1K_TESTSIM_STRICT', CONFIG.get('model_id')
assert MODEL_INFO.get('model_id') == 'V1_TL_1K_TESTSIM_STRICT', MODEL_INFO.get('model_id')
assert MODEL_INFO.get('freeze_status') == 'FROZEN', MODEL_INFO.get('freeze_status')
assert aggregator_name == 'MOST_CONFIDENT_SPECTRUM', aggregator_name

ACCEL = detect_accelerator()
print('bundle_version:', CONFIG.get('bundle_version'))
print('manifest version:', MANIFEST.get('bundle_version'))
print('model:', CONFIG.get('model_id'))
print('aggregator:', aggregator_name)
print('temperature:', CONFIG.get('calibration_temperature'))
print('GPU:', ACCEL.get('gpu_name'))


bundle_version: v2-A7
manifest version: v2-A7-60174e39a2a3c6b4-V1_TL_1K_TESTSIM_STRICT
model: V1_TL_1K_TESTSIM_STRICT
aggregator: MOST_CONFIDENT_SPECTRUM
temperature: 1.1947045372735254
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [5]:
# 5. Runtime configuration — explicit named Drive files, local output only
CFG = RuntimeConfig(
    runtime='colab',
    input_root=COMPETITION_ROOT,
    work_dir=WORK_DIR,
    device_preference=DEVICE_PREFERENCE,
    require_gpu=REQUIRE_GPU,
    expected_config_hash=None,
    extra={
        'github_commit': REPO_COMMIT,
        'input_mode': 'FILENAMES_ONLY',
        'bundle_root': str(BUNDLE_ROOT),
        'test_file': TEST_PATH.name,
        'sample_submission_file': SAMPLE_SUBMISSION_PATH.name,
    },
)
print(json.dumps(CFG.as_dict(), indent=2, default=str))


{
  "runtime": "colab",
  "input_root": "/content/drive/MyDrive/EnvedaCASMI/competition",
  "work_dir": "/content/casmi_runtime/output/20260929T041309Z",
  "device_preference": "gpu",
  "require_gpu": false,
  "expected_model_id": "V1_TL_1K_TESTSIM_STRICT",
  "expected_aggregator": "MOST_CONFIDENT_SPECTRUM",
  "expected_bundle_version": "v2-A7",
  "expected_config_hash": null,
  "time_budget_s": 28800,
  "emergency_cap_enabled": false,
  "emergency_cap_n": 300,
  "extra": {
    "github_commit": "9550ca849f0f21c6d4d09bb7af33582d0dda4eff",
    "input_mode": "FILENAMES_ONLY",
    "bundle_root": "/content/drive/MyDrive/EnvedaCASMI/bundle",
    "test_file": "test.parquet",
    "sample_submission_file": "sample_submission.csv"
  }
}


In [6]:
# 6. Functional gate + full inference
# Current migration policy: feature-only fixture drift is accepted only when every downstream
# invariant is exact: candidate set, fold scores, final scores, ranks, probabilities,
# selected spectrum, confidence, and final molecule ranking.
REPORT = run_named_inference(
    CFG,
    bundle_dir=BUNDLE_ROOT,
    test_path=TEST_PATH,
    sample_path=SAMPLE_SUBMISSION_PATH,
    allow_feature_only_drift=True,
)


[runtime] colab | device cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)
[bundle] v2-A7 | CONFIG_HASH 60174e39a2a3c6b4 | model V1_TL_1K_TESTSIM_STRICT | aggregator MOST_CONFIDENT_SPECTRUM (T=1.1947045372735254) | filenames only
BACKEND FALLBACK: GPU not used: the frozen bundle code (v2-A7) implements no GPU backend; GPU spectral evidence needs a parity-validated implementation shipped in a re-exported bundle -> using numba
[self-test] PASS on numba | strict=False | feature_mismatches=3 | downstream_exact=True
[infer] 50/1213 spectra  22s  rss=3.2GB
[infer] 101/1213 spectra  40s  rss=3.4GB
[infer] 151/1213 spectra  54s  rss=3.6GB
[infer] 203/1213 spectra  63s  rss=3.7GB
[infer] 251/1213 spectra  71s  rss=3.8GB
[infer] 302/1213 spectra  82s  rss=3.8GB
[infer] 352/1213 spectra  95s  rss=3.9GB
[infer] 404/1213 spectra  109s  rss=3.9GB
[infer] 450/1213 spectra  122s  rss=3.9GB
[infer] 502/1213 spectra  133s  rss=3.9GB
[infer] 550/1213 spectra  140s  rss=3.9GB
[infer] 600/1213 spectra  1

In [7]:
# 7. Validate the produced report and inspect submission
import pandas as pd

assert REPORT['bundle_integrity_mode'] == 'FILENAMES_ONLY'
assert REPORT['model_id'] == 'V1_TL_1K_TESTSIM_STRICT'
assert REPORT['aggregator'] == 'MOST_CONFIDENT_SPECTRUM'
assert REPORT['self_test_status'] == 'PASS'
assert REPORT['self_test_downstream_exact'] is True
assert REPORT['submission_validation_status'] == 'PASS'
assert Path(REPORT['casmi_infer_source']).resolve().is_relative_to((BUNDLE_ROOT / 'code').resolve())

keys = [
    'bundle_integrity_mode', 'bundle_version', 'model_id', 'aggregator',
    'calibration_temperature', 'actual_backend', 'gpu_used', 'fallback_reason',
    'self_test_status', 'self_test_strict_passed', 'self_test_feature_mismatches',
    'self_test_downstream_exact', 'self_test_feature_only_drift_accepted',
    'n_spectra', 'n_molecules', 'candidate_pair_count', 'runtime_seconds',
    'submission_rows', 'submission_validation_status', 'casmi_infer_source',
]
print(json.dumps({k: REPORT.get(k) for k in keys}, indent=2, default=str))

submission = pd.read_csv(WORK_DIR / 'submission.csv')
print('submission shape:', submission.shape)
display(submission.head(3))


{
  "bundle_integrity_mode": "FILENAMES_ONLY",
  "bundle_version": "v2-A7",
  "model_id": "V1_TL_1K_TESTSIM_STRICT",
  "aggregator": "MOST_CONFIDENT_SPECTRUM",
  "calibration_temperature": 1.1947045372735254,
  "actual_backend": "numba",
  "gpu_used": false,
  "fallback_reason": "GPU not used: the frozen bundle code (v2-A7) implements no GPU backend; GPU spectral evidence needs a parity-validated implementation shipped in a re-exported bundle",
  "self_test_status": "PASS",
  "self_test_strict_passed": false,
  "self_test_feature_mismatches": 3,
  "self_test_downstream_exact": true,
  "self_test_feature_only_drift_accepted": true,
  "n_spectra": 1213,
  "n_molecules": 400,
  "candidate_pair_count": 471173,
  "runtime_seconds": 466.2886257171631,
  "submission_rows": 400,
  "submission_validation_status": "PASS",
  "casmi_infer_source": "/content/drive/MyDrive/EnvedaCASMI/bundle/code/casmi_infer"
}
submission shape: (400, 2)


,molecule_id,smiles
0,m_005e53,CN(Cc1ncccc1C(=O)O)CC1(c2ccc(Br)cc2)CC1;O=C(O)...
1,m_006153,COc1ccc(N2CC(C(=O)Nc3nncn3C3CC3)CC2=O)cc1;Cc1n...
2,m_00b5aa,COc1ccnc(CN2CCOCC3(CCCC3)C2)c1OC;CC(C)(C)c1cc(...


In [8]:
# 8. Save results to a new Drive folder by filename only
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DEST = DRIVE_RESULTS / stamp
DEST.mkdir(parents=True, exist_ok=False)

for name in ('submission.csv', 'run_report.json'):
    src = WORK_DIR / name
    if not src.exists():
        raise FileNotFoundError(src)
    shutil.copy2(src, DEST / name)

provenance = {
    'created_at_utc': stamp,
    'repo_url': REPO_URL,
    'repo_commit': REPO_COMMIT,
    'bundle_root': str(BUNDLE_ROOT),
    'bundle_version': CONFIG.get('bundle_version'),
    'model_id': CONFIG.get('model_id'),
    'aggregator': aggregator_name,
    'input_mode': 'FILENAMES_ONLY',
    'bundle_file': 'manifest.json',
    'config_file': 'config.json',
    'model_info_file': 'models/model_info.json',
    'test_file': TEST_PATH.name,
    'sample_submission_file': SAMPLE_SUBMISSION_PATH.name,
    'output_files': ['submission.csv', 'run_report.json'],
}
(DEST / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')

print('Saved:', DEST)
print('Files:', sorted(p.name for p in DEST.iterdir()))


Saved: /content/drive/MyDrive/EnvedaCASMI/results/20260929T042103Z
Files: ['provenance.json', 'run_report.json', 'submission.csv']


## Finished

The generated `submission.csv` is still submitted to Kaggle manually. Review `run_report.json` first, especially the backend, functional self-test fields, fallback counts, and submission validation status.
